In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Install Required Libraries

In [ ]:
!pip -q install segmentation-models-pytorch
!pip -q install albumentations
!pip -q install opencv-python
!pip -q install tqdm
!pip -q install scikit-learn

Import Libraries

In [ ]:
import os
import cv2
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from glob import glob
from tqdm import tqdm

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")

Set Random Seed

In [ ]:
SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

GPU Information

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*50)
print("PyTorch Version :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("="*50)

Project Configuration

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/DIP_PROJECT"

IMAGE_DIR = os.path.join(PROJECT_ROOT, "dataset/images")

MASK_DIR = os.path.join(PROJECT_ROOT, "dataset/masks")

MODEL_DIR = os.path.join(PROJECT_ROOT, "models")

RESULT_DIR = os.path.join(PROJECT_ROOT, "results")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

Load Dataset

In [ ]:
image_paths = sorted(
    glob(os.path.join(IMAGE_DIR, "*.jpg"))
)

mask_paths = sorted(
    glob(os.path.join(MASK_DIR, "*.png"))
)

print(f"Total Images : {len(image_paths)}")
print(f"Total Masks  : {len(mask_paths)}")

Verify Dataset

In [ ]:
assert len(image_paths) == len(mask_paths), "Mismatch between images and masks."

print("Dataset verified successfully!")

Create DataFrame

In [ ]:
df = pd.DataFrame({

    "image": image_paths,

    "mask": mask_paths

})

df.head()

Train / Validation / Test Split

In [ ]:
train_df, test_df = train_test_split(

    df,

    test_size=0.15,

    random_state=SEED,

    shuffle=True

)

train_df, val_df = train_test_split(

    train_df,

    test_size=0.1765,

    random_state=SEED,

    shuffle=True

)

print("Training Images :", len(train_df))
print("Validation Images :", len(val_df))
print("Testing Images :", len(test_df))

Save splits

In [ ]:
train_df.to_csv(

    os.path.join(PROJECT_ROOT,"train.csv"),

    index=False

)

val_df.to_csv(

    os.path.join(PROJECT_ROOT,"val.csv"),

    index=False

)

test_df.to_csv(

    os.path.join(PROJECT_ROOT,"test.csv"),

    index=False

)

print("CSV files saved.")

Visualize Sample

In [ ]:
sample = df.sample(1).iloc[0]

image = cv2.imread(sample["image"])

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

mask = cv2.imread(sample["mask"],0)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.imshow(image)

plt.title("Original Image")

plt.axis("off")

plt.subplot(1,2,2)

plt.imshow(mask,cmap="gray")

plt.title("Ground Truth Mask")

plt.axis("off")

plt.show()

Helper Functions

Image Preprocessing

In [ ]:
IMG_SIZE = 256

def resize_image(image):
    return cv2.resize(
        image,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_AREA
    )


def rgb2gray(image):
    return cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)


def remove_hair(gray):

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (9,9)
    )

    blackhat = cv2.morphologyEx(
        gray,
        cv2.MORPH_BLACKHAT,
        kernel
    )

    _, hair = cv2.threshold(
        blackhat,
        10,
        255,
        cv2.THRESH_BINARY
    )

    gray = cv2.inpaint(
        gray,
        hair,
        3,
        cv2.INPAINT_TELEA
    )

    return gray


def preprocess(image):

    image = resize_image(image)

    gray = rgb2gray(image)

    # Hair removal
    gray = remove_hair(gray)

    # CLAHE
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    gray = clahe.apply(gray)

    # Median preserves edges
    blur = cv2.medianBlur(gray,5)

    return image, gray, blur

Verify Preprocessing

In [ ]:
img,gray,blur = preprocess(image)

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)

plt.imshow(img)

plt.title("RGB")

plt.axis("off")

plt.subplot(1,3,2)

plt.imshow(gray,cmap="gray")

plt.title("Gray")

plt.axis("off")

plt.subplot(1,3,3)

plt.imshow(blur,cmap="gray")

plt.title("Gaussian Blur")

plt.axis("off")

plt.show()

Otsu Segmentation

In [ ]:
def otsu_segment(image):

    image, gray, blur = preprocess(image)

    _, mask = cv2.threshold(
        blur,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5,5)
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

    if num_labels > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = (labels == largest).astype(np.uint8)

    return mask

Test Otsu

In [ ]:
sample = df.sample(1).iloc[0]

img = cv2.imread(sample["image"])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

mask = otsu_segment(img)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap="gray")
plt.title("Otsu Result")
plt.axis("off")

plt.show()

K-means

In [ ]:
def kmeans_segment(image):

    image = resize_image(image)

    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    pixels = lab.reshape((-1,3)).astype(np.float32)

    kmeans = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(pixels)

    centers = kmeans.cluster_centers_

    lesion_cluster = np.argmin(centers[:,0])

    mask = (labels == lesion_cluster).reshape(
        IMG_SIZE,
        IMG_SIZE
    ).astype(np.uint8)

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5,5)
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

    if num_labels > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = (labels == largest).astype(np.uint8)

    return mask

Test K-means

In [ ]:
mask = kmeans_segment(img)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap="gray")
plt.title("K-Means Result")
plt.axis("off")

plt.show()

Watershed Segmentation

In [ ]:
def watershed_segment(image):

    image, gray, blur = preprocess(image)

    _, thresh = cv2.threshold(
        blur,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5,5)
    )

    opening = cv2.morphologyEx(
        thresh,
        cv2.MORPH_OPEN,
        kernel,
        iterations=3
    )

    sure_bg = cv2.dilate(
        opening,
        kernel,
        iterations=3
    )

    dist = cv2.distanceTransform(
        opening,
        cv2.DIST_L2,
        5
    )

    _, sure_fg = cv2.threshold(
        dist,
        0.4 * dist.max(),
        255,
        0
    )

    sure_fg = np.uint8(sure_fg)

    unknown = cv2.subtract(
        sure_bg,
        sure_fg
    )

    _, markers = cv2.connectedComponents(
        sure_fg
    )

    markers += 1

    markers[unknown == 255] = 0

    markers = cv2.watershed(image, markers)

    mask = (markers > 1).astype(np.uint8)

    return mask

Test Watershed

In [ ]:
mask = watershed_segment(img)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap="gray")
plt.title("Watershed Result")
plt.axis("off")

plt.show()

Canny + Contour Filling

In [ ]:
def canny_segment(image):

    image, gray, blur = preprocess(image)

    edges = cv2.Canny(
        blur,
        40,
        120
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5,5)
    )

    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    mask = np.zeros(gray.shape, np.uint8)

    if len(contours):

        largest = max(
            contours,
            key=cv2.contourArea
        )

        cv2.drawContours(
            mask,
            [largest],
            -1,
            255,
            thickness=cv2.FILLED
        )

    mask = (mask > 127).astype(np.uint8)

    return mask

Test Canny

In [ ]:
mask = canny_segment(img)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap="gray")
plt.title("Canny Result")
plt.axis("off")

plt.show()

In [ ]:
from skimage.segmentation import chan_vese
import cv2
import numpy as np

def chanvese_segment(image):

    image, gray, blur = preprocess(image)

    # Lighter smoothing — heavy blur here washes out the already-low contrast
    blur = cv2.GaussianBlur(blur, (5, 5), 0)

    # Contrast stretch so lesion/background separation is stronger
    # before Chan-Vese sees it
    blur = cv2.normalize(blur, None, 0, 255, cv2.NORM_MINMAX)

    img = blur.astype(np.float64) / 255.0

    mask = chan_vese(
        img,
        mu=0.15,                 # lower mu = less curvature penalty, lets contour move more
        lambda1=1,
        lambda2=1,               # balanced — lambda2=2 was biasing against shrinking
        tol=1e-6,                # tighter tolerance so it doesn't stop early
        max_num_iter=2000,       # more room to converge
        dt=0.5,
        init_level_set="checkerboard",  # avoids getting stuck near a disk-shaped minimum
        extended_output=False
    )

    mask = mask.astype(np.uint8)

    # Guard against degenerate masks (all foreground or all background)
    fg = gray[mask == 1]
    bg = gray[mask == 0]

    if fg.size == 0 or bg.size == 0:
        return mask

    # Keep the darker region as lesion
    if fg.mean() > bg.mean():
        mask = 1 - mask

    # Fill holes and smooth edges
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return mask

In [ ]:
mask = chanvese_segment(img)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap="gray")
plt.title("Chan-Vese Result")
plt.axis("off")

plt.show()

Unified Method Dictionary

In [ ]:
METHODS = {
    "Otsu": otsu_segment,
    "KMeans": kmeans_segment,
    "Watershed": watershed_segment,
    "Canny": canny_segment,
    "ChanVese": chanvese_segment
}

Compare All Classical Methods

In [ ]:
sample = df.sample(1).iloc[0]

img = cv2.imread(sample["image"])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

gt = cv2.imread(sample["mask"], 0)
gt = cv2.resize(gt, (IMG_SIZE, IMG_SIZE))
gt = (gt > 127).astype(np.uint8)

plt.figure(figsize=(15,8))

plt.subplot(2,4,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(2,4,2)
plt.imshow(gt, cmap="gray")
plt.title("Ground Truth")
plt.axis("off")

i = 3

for name, func in METHODS.items():

    mask = func(img)

    plt.subplot(2,4,i)
    plt.imshow(mask, cmap="gray")
    plt.title(name)
    plt.axis("off")

    i += 1

plt.tight_layout()
plt.show()

Helper: Flatten Masks

In [ ]:
def flatten(mask):

    return mask.reshape(-1)

Confusion Matrix Components

In [ ]:
def confusion_components(pred, gt):

    pred = flatten(pred)
    gt = flatten(gt)

    TP = np.sum((pred == 1) & (gt == 1))
    TN = np.sum((pred == 0) & (gt == 0))
    FP = np.sum((pred == 1) & (gt == 0))
    FN = np.sum((pred == 0) & (gt == 1))

    return TP, TN, FP, FN

In [ ]:
def iou(TP, FP, FN):

    return TP / (TP + FP + FN + 1e-8)

def dice(TP, FP, FN):

    return (2 * TP) / (2 * TP + FP + FN + 1e-8)

def accuracy(TP, TN, FP, FN):

    return (TP + TN) / (TP + TN + FP + FN + 1e-8)

def precision(TP, FP):

    return TP / (TP + FP + 1e-8)

def recall(TP, FN):

    return TP / (TP + FN + 1e-8)

Metrics Functions

Single Image Evaluation Function

In [ ]:
def evaluate_one(image, gt, method_func):

    pred = method_func(image)

    TP, TN, FP, FN = confusion_components(pred, gt)

    return {

        "IoU": iou(TP, FP, FN),

        "Dice": dice(TP, FP, FN),

        "Accuracy": accuracy(TP, TN, FP, FN),

        "Precision": precision(TP, FP),

        "Recall": recall(TP, FN)
    }

Run Evaluation on Test Set

In [ ]:
results = []

for row in tqdm(test_df.itertuples(index=False), total=len(test_df)):

    image = cv2.imread(row.image)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    gt = cv2.imread(row.mask, 0)
    gt = cv2.resize(gt, (IMG_SIZE, IMG_SIZE))
    gt = (gt > 127).astype(np.uint8)

    for name, method in METHODS.items():
        scores = evaluate_one(image, gt, method)
        scores["Method"] = name
        results.append(scores)

Convert to Dataframe

In [ ]:
results_df = pd.DataFrame(results)

results_df

Save Results

In [ ]:
results_df.to_csv(
    os.path.join(RESULT_DIR, "segmentation_results.csv"),
    index=False
)

print("Results saved successfully.")

Summary Table (Mean Scores)

In [ ]:
summary = results_df.groupby("Method").mean()

summary

Visualization

In [ ]:
plt.figure(figsize=(8,5))

summary["IoU"].plot(kind="bar")

plt.title("IoU Comparison")

plt.ylabel("IoU Score")

plt.grid()

plt.show()

plt.figure(figsize=(8,5))

summary["Dice"].plot(kind="bar")

plt.title("Dice Score Comparison")

plt.ylabel("Dice Score")

plt.grid()

plt.show()

Sorting

In [ ]:
summary.sort_values("Dice", ascending=False)

In [ ]:
train_transform = A.Compose([

    A.Resize(256,256),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    A.RandomBrightnessContrast(p=0.3),

    A.Normalize(),

    ToTensorV2()

])

val_transform = A.Compose([

    A.Resize(256,256),

    A.Normalize(),

    ToTensorV2()

])

In [ ]:
class ISICDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        image = cv2.imread(self.df.iloc[idx]["image"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.df.iloc[idx]["mask"], 0)

        mask = (mask > 127).astype(np.float32)

        if self.transform:

            augmented = self.transform(
                image=image,
                mask=mask
            )

            image = augmented["image"]
            mask = augmented["mask"]

        # IMPORTANT FIX (from earlier audit)
        if isinstance(mask, np.ndarray):
            mask = torch.tensor(mask, dtype=torch.float32)

        mask = mask.unsqueeze(0)

        return image.float(), mask

In [ ]:
train_loader = DataLoader(

    ISICDataset(train_df, train_transform),

    batch_size=8,
    shuffle=True,
    num_workers=2

)

val_loader = DataLoader(

    ISICDataset(val_df, val_transform),

    batch_size=8,
    shuffle=False,
    num_workers=2

)

In [ ]:
images, masks = next(iter(train_loader))

print(images.shape)   # should be [8,3,256,256]
print(masks.shape)    # should be [8,1,256,256]

U-Net Model Definition

In [ ]:
model = smp.Unet(

    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None

)

model = model.to(device)

print("U-Net model created successfully")

Loss Function

In [ ]:
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)
bce_loss = nn.BCEWithLogitsLoss()

def loss_fn(pred, target):

    return dice_loss(pred, target) + bce_loss(pred, target)

Optimizer + Scheduler

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,
    mode="min",
    factor=0.5,
    patience=2

)

In [ ]:
train_losses = []
val_losses = []

best_loss = np.inf
patience = 5
counter = 0

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):

    model.train()
    running_loss = 0

    for images, masks in tqdm(loader):

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = loss_fn(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
def validate(model, loader, loss_fn):

    model.eval()
    running_loss = 0

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = loss_fn(outputs, masks)

            running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        loss_fn
    )

    val_loss = validate(
        model,
        val_loader,
        loss_fn
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss  : {val_loss:.4f}")

    # Save best model
    if val_loss < best_loss:

        best_loss = val_loss

        torch.save(
            model.state_dict(),
            os.path.join(MODEL_DIR, "best_unet.pth")
        )

        counter = 0

        print("Best model saved!")

    else:

        counter += 1

        print(f"Early stopping counter: {counter}/{patience}")

    if counter >= patience:

        print("Early stopping triggered")
        break

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.title("U-Net Training Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.grid()
plt.show()

In [ ]:
train_plot = train_losses[:10]
val_plot = val_losses[:10]

plt.figure(figsize=(8,5))

plt.plot(train_plot, label="Train Loss")
plt.plot(val_plot, label="Validation Loss")

plt.title("U-Net Training Curve (First 10 Epochs)")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.xticks(range(10), range(1, 11))  # Show epochs 1–10

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        os.path.join(MODEL_DIR, "best_unet.pth"),
        map_location=device
    )
)

model.eval()

print("Best U-Net model loaded")

In [ ]:
def unet_segment(image):

    image = resize_image(image)

    image = image.astype(np.float32) / 255.0

    image = np.transpose(image, (2,0,1))

    image = torch.tensor(image).unsqueeze(0).to(device)

    with torch.no_grad():

        pred = model(image)

        pred = torch.sigmoid(pred)

    pred = pred.squeeze().cpu().numpy()

    pred = (pred > 0.5).astype(np.uint8)

    return pred

In [ ]:
METHODS["UNet"] = unet_segment

print("U-Net added to evaluation pipeline")

In [ ]:
sample = df.sample(1).iloc[0]

img = cv2.imread(sample["image"])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

pred = unet_segment(img)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(pred, cmap="gray")
plt.title("U-Net Prediction")
plt.axis("off")

plt.show()

In [ ]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):

    image = cv2.imread(row["image"])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    gt = cv2.imread(row["mask"], cv2.IMREAD_GRAYSCALE)
    gt = cv2.resize(gt, (IMG_SIZE, IMG_SIZE))
    gt = (gt > 127).astype(np.uint8)

    scores = evaluate_one(image, gt, unet_segment)
    scores["Method"] = "U-Net"

    results.append(scores)

results_df = pd.DataFrame(results)
print(results_df.mean(numeric_only=True))
results_df.to_csv("unet_results.csv", index=False)

In [ ]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):

    image = cv2.imread(row["image"])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    gt = cv2.imread(row["mask"], 0)
    gt = cv2.resize(gt, (IMG_SIZE, IMG_SIZE))
    gt = (gt > 127).astype(np.uint8)

    for name, method in METHODS.items():

        scores = evaluate_one(image, gt, method)

        scores["Method"] = name

        results.append(scores)

In [ ]:
results_df = pd.DataFrame(results)

results_df.head()

In [ ]:
results_df.to_csv(
    os.path.join(RESULT_DIR, "all_results.csv"),
    index=False
)

print("Raw results saved")

In [ ]:
summary = results_df.groupby("Method").mean()

summary

In [ ]:
summary.sort_values("Dice", ascending=False)

In [ ]:
plt.figure(figsize=(8,5))

summary["IoU"].plot(kind="bar")

plt.title("IoU Comparison (All Methods)")
plt.ylabel("IoU Score")
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

summary["Dice"].plot(kind="bar")

plt.title("Dice Score Comparison (All Methods)")
plt.ylabel("Dice Score")
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

summary["Accuracy"].plot(kind="bar")

plt.title("Accuracy Comparison")
plt.ylabel("Accuracy")
plt.grid()
plt.show()

In [ ]:
summary[["Precision","Recall"]].plot(kind="bar", figsize=(8,5))

plt.title("Precision vs Recall")
plt.grid()
plt.show()

In [ ]:
sample = df.sample(1).iloc[0]

image = cv2.imread(sample["image"])
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

gt = cv2.imread(sample["mask"], 0)
gt = cv2.resize(gt, (IMG_SIZE, IMG_SIZE))
gt = (gt > 127).astype(np.uint8)

plt.figure(figsize=(15,8))

plt.subplot(2,3,1)
plt.imshow(image)
plt.title("Original")
plt.axis("off")

plt.subplot(2,3,2)
plt.imshow(gt, cmap="gray")
plt.title("Ground Truth")
plt.axis("off")

i = 1

for name, method in METHODS.items():

    pred = method(image)

    plt.subplot(2,3,i)
    plt.imshow(pred, cmap="gray")
    plt.title(name)
    plt.axis("off")

    i += 1

plt.tight_layout()
plt.show()

In [ ]:
summary.to_csv(
    os.path.join(RESULT_DIR, "summary_metrics.csv")
)

print("Summary saved")